# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule in plain words:** flag a page for CTR review if it sits in a good position tier
(top_3, page_1, or striking), has enough impression volume to trust the CTR estimate,
and its CTR falls meaningfully below the average CTR of other pages at its own tier.
Bigger the gap and bigger the volume, higher the score, since a big gap on a
high-traffic page is worth more editor time than the same gap on a rarely-seen page.

**Reason code:** one code per row, `ctr_gap_vs_tier`, since this baseline tests exactly
one hypothesis. A future rule could add `stale_and_visible` or `quick_win_volume`
as separate codes, but this week is one rule, one reason.

**Action label:** `review_title_snippet` for anything that scores above zero,
`no_action` otherwise.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError("HF_TOKEN environment variable not set.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

agg = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_prev,
        SUM(gsc_clicks) AS clicks_prev,
        AVG(gsc_avg_position) AS avg_position_prev
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date < DATE '2026-03-16'
    GROUP BY client_hash_id, content_hash_id
""").df()

agg["ctr_prev"] = agg["clicks_prev"] / agg["impressions_prev"].replace(0, np.nan)
df = agg[agg["impressions_prev"] >= 100].dropna(subset=["ctr_prev"]).copy()
print(f"Pages with ≥100 impressions in first half: {df.shape[0]}")

Pages with ≥100 impressions in first half: 77540


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
bins = [0, 3, 10, 20, 50, np.inf]
labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
df["position_tier"] = pd.cut(df["avg_position_prev"], bins=bins, labels=labels)

df["tier_avg_ctr"] = df.groupby("position_tier", observed=True)["ctr_prev"].transform("mean")
df["ctr_gap"] = df["tier_avg_ctr"] - df["ctr_prev"]

good_tier = df["position_tier"].isin(["top_3", "page_1", "striking"]).astype(int)
positive_gap = (df["ctr_gap"] > 0).astype(int)
df["score"] = good_tier * positive_gap * df["ctr_gap"] * df["impressions_prev"]

df["reason_code"] = "ctr_gap_vs_tier"
df["action"] = np.where(df["score"] > 0, "review_title_snippet", "no_action")

queue = df.sort_values("score", ascending=False)[
    ["content_hash_id", "client_hash_id", "position_tier", "impressions_prev",
     "ctr_prev", "tier_avg_ctr", "ctr_gap", "score", "reason_code", "action"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue written: {queue.shape[0]} rows")
print(f"Flagged for review: {(queue['action'] == 'review_title_snippet').sum()}")
print(queue.head(10))

Queue written: 77540 rows
Flagged for review: 40369
                 content_hash_id           client_hash_id position_tier  \
86715   content_9c057b66c30a3abb  client_73cda7b4e4f265ea        page_1   
92534   content_34a70fea29d15f24  client_62f4a7e64f5e0096         top_3   
5191    content_7c6373141eae744a  client_62f4a7e64f5e0096        page_1   
114129  content_8e1334d6356668e3  client_73cda7b4e4f265ea        page_1   
126718  content_65c75874a23fca87  client_23a62021009f63c4        page_1   
54781   content_945d6ff91386c817  client_62f4a7e64f5e0096        page_1   
150321  content_82e35c4845e6c391  client_20259bd6705d81d4      striking   
116855  content_f6116743b00afc2d  client_62f4a7e64f5e0096        page_1   
111564  content_1642f339bd6e7c8d  client_62f4a7e64f5e0096        page_1   
138258  content_acbcc847f8996314  client_62f4a7e64f5e0096        page_1   

        impressions_prev  ctr_prev  tier_avg_ctr   ctr_gap       score  \
86715            83772.0  0.000000      0.003349

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).reset_index(drop=True)

for i, row in top20.iterrows():
    print(f"{i+1}. {row['action']} | content_id={row['content_hash_id']} | tier={row['position_tier']} "
          f"| impressions={row['impressions_prev']:.0f} | ctr={row['ctr_prev']:.4f} vs tier avg {row['tier_avg_ctr']:.4f}")
    print(f"   why: gap of {row['ctr_gap']:.4f} points below its tier average, "
          f"on {row['impressions_prev']:.0f} impressions, score {row['score']:.1f}")
    print(f"   would be wrong if: the low CTR is normal for this page's query intent "
          f"(e.g. informational, no click-through expected), or a SERP feature is stealing clicks unrelated to the title")
    print()

1. review_title_snippet | content_id=content_9c057b66c30a3abb | tier=page_1 | impressions=83772 | ctr=0.0000 vs tier avg 0.0033
   why: gap of 0.0033 points below its tier average, on 83772 impressions, score 280.5
   would be wrong if: the low CTR is normal for this page's query intent (e.g. informational, no click-through expected), or a SERP feature is stealing clicks unrelated to the title

2. review_title_snippet | content_id=content_34a70fea29d15f24 | tier=top_3 | impressions=73639 | ctr=0.0002 vs tier avg 0.0040
   why: gap of 0.0038 points below its tier average, on 73639 impressions, score 279.8
   would be wrong if: the low CTR is normal for this page's query intent (e.g. informational, no click-through expected), or a SERP feature is stealing clicks unrelated to the title

3. review_title_snippet | content_id=content_7c6373141eae744a | tier=page_1 | impressions=86860 | ctr=0.0006 vs tier avg 0.0033
   why: gap of 0.0028 points below its tier average, on 86860 impressions, sc

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak = top20[top20["impressions_prev"] < top20["impressions_prev"].median()]
print("Weakest picks in the top 20, lower relative volume than their peers:")
print(weak[["content_hash_id", "position_tier", "impressions_prev", "ctr_gap", "score"]])

print("\nWhy weak: even above the 100-impression floor, a page near that floor has a shakier "
      "CTR estimate than one with thousands of impressions, so its rank could shuffle on a noisy day.")

print("\nFeatures used in score: ['position_tier', 'impressions_prev', 'ctr_prev', 'tier_avg_ctr']")
print("All are computed exclusively from the first 15 days of March – no future window or label-derived column is involved.")

Weakest picks in the top 20, lower relative volume than their peers:
             content_hash_id position_tier  impressions_prev   ctr_gap  \
5   content_945d6ff91386c817        page_1           49314.0  0.003308   
10  content_36fc1ee501ec072d        page_1           46199.0  0.003111   
11  content_62673eea26c31c17        page_1           49386.0  0.002640   
13  content_306bc78dff1eb683         top_3           33020.0  0.003621   
14  content_09b05ffb4d7f6c8d        page_1           41590.0  0.002724   
15  content_6a9c79f55413b447         top_3           41220.0  0.002735   
16  content_cd3d932d4e1c8db0        page_1           34191.0  0.003290   
17  content_cb6d4179551785be         top_3           43871.0  0.002472   
18  content_0adb360f9005b515        page_1           40178.0  0.002527   
19  content_252aa5480bb1f8d7         top_3           31460.0  0.003155   

         score  
5   163.141215  
10  143.709799  
11  130.382327  
13  119.555434  
14  113.275320  
15  112.721835

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.